## fix diabetes type 1&2 issue

In [15]:
import json
import pandas as pd
from transformers import pipeline
import spacy
from scispacy.linking import EntityLinker
import gdown 
import re

nlp = spacy.load("en_core_sci_sm")  
print("OK:", nlp.pipe_names)

ner = pipeline(
    "token-classification",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="max"  # better merging of wordpieces
)

# Download train, validation and test splits from Google Drive
#gdown.download("https://drive.google.com/uc?export=download&id=1ria4E6IdTIPsikL4Glm3uy1tFKJKw0W8", "train.json", quiet=False)
#gdown.download("https://drive.google.com/uc?export=download&id=1KAZneuwdfEVQQM6euCX4pMDP-9DQpiB5", "validation.json", quiet=False)
#gdown.download("https://drive.google.com/uc?export=download&id=10izqL71kcgnteYsf87Vh6j_mZ8sZM2Rc", "test.json", quiet=False)

# Load training data into memory
with open("train.json", "r") as f:
    train_data = json.load(f)

print(f"Total training examples: {len(train_data)}")

rows = []
negation_keywords = ["no ", "not ", "denies ", "without ", "free of ", "negative for ", "don't"] 

for i, item in enumerate(train_data[:]):
    for turn_idx, utterance in enumerate(item["utterances"]):

        # skip empty turns
        if not utterance.strip():
            continue

        entities = ner(utterance)

        for ent in entities:

            # =========================
            # Begin of Patch 1: fix truncated diabetes terms
            # Example:
            # "type 1 diabetes" should not become only "diabetes"
            # =========================
            if ent["word"].strip().lower() == "diabetes":
                prefix = utterance[max(0, ent["start"] - 30):ent["start"]]

                match = re.search(
                    r"(type\s*(?:1|2|one|two|i|ii))\s*$",
                    prefix,
                    re.IGNORECASE
                )

                if match:
                    new_start = ent["start"] - len(match.group(1)) - 1
                    ent["start"] = new_start
                    ent["word"] = utterance[ent["start"]:ent["end"]]
            # =========================
            # End of Patch 1: fix truncated diabetes terms
            # =========================
            current_word = ent["word"].lower().strip()
            words_in_entity = current_word.split()
            
            # Connect structurally unfinished words
            dangling_connectors = ["of", "in", "with", "to", "on", "and", "or", "for", "the"]
            
            # Medical adjectives that often miss the term next to it
            orphaned_adjectives = [
                "dry", "tight", "sharp", "dull", "severe", "mild", 
                "chronic", "acute", "throbbing", "stabbing", "burning", 
                "heavy", "persistent", "productive", "swollen"
            ]
            
            if words_in_entity:
                last_word = words_in_entity[-1]
                
                if last_word in dangling_connectors or last_word in orphaned_adjectives:
                    text_after = utterance[ent["end"]:]
                    
                    match = re.match(r"\s+([a-zA-Z0-9\-]+)", text_after)
                    
                    if match:
                        ent["end"] += len(match.group(0))
                        ent["word"] = utterance[ent["start"]:ent["end"]]

            start_idx = ent["start"]
            
            # Check 15 characters before the entity for negation
            context_window = utterance[max(0, start_idx - 15):start_idx].lower()
            is_negated = any(neg_word in context_window for neg_word in negation_keywords)
            
            if is_negated:
                continue

            rows.append({
                "dialogue_id":   i,
                "description":   item["description"],
                "turn_idx":      turn_idx,
                "original_term": ent["word"],
                "entity_type":   ent["entity_group"],
                "score":         round(ent["score"], 3),
                "char_start":    ent["start"],
                "char_end":      ent["end"],
                "turn_text":     utterance,
            })

import pandas as pd
df = pd.DataFrame(rows)

# keep only clinical entity types
keep = {"Disease_disorder", "Medication", "Sign_symptom"}
df = df[df['entity_type'].isin(keep)]

# optional: drop low confidence
df = df[df['score'] >= 0.7]

df.to_csv("entities.csv", index=False)
print(f"Found {len(df)} entities across {df['dialogue_id'].nunique()} dialogues")
df.head(20)
df.drop(columns=['turn_text', 'description']).to_csv("entities_clean.csv", index=False)



OK: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner']


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 5116.84it/s]


Total training examples: 482
Found 2473 entities across 443 dialogues


## test functionality

In [23]:
df[
    df["turn_text"].str.contains(
        "type 1 diabetes",
        case=False,
        na=False
    )
][
    ["original_term", "turn_text"]
].assign(
    turn_text=lambda x: x["turn_text"].apply(
        lambda s: s[:100] + "..." if len(s) > 200 else s
    )
)

,original_term,turn_text
3742,symptoms,"patient: what are my chances of only getting mild symptoms if i get the virus, because i have type 1 diabetes?"
3968,unwell,"patient: what are my chances of becoming severely unwell with the coronavirus, as i have type 1 diabetes?"
5880,anxious state,"patient: i have type 1 diabetes and to put it simply, i am living in an anxious state right now, wor..."
6178,diabetes,"doctor: in brief: diabetes type 1 diabetes may be an autoimmune disorder, but if well controlled one..."
6179,type 1 diabetes,"doctor: in brief: diabetes type 1 diabetes may be an autoimmune disorder, but if well controlled one..."
6182,infections,"doctor: in brief: diabetes type 1 diabetes may be an autoimmune disorder, but if well controlled one..."
6184,kidney disease,"doctor: in brief: diabetes type 1 diabetes may be an autoimmune disorder, but if well controlled one..."
6185,dehydration,"doctor: in brief: diabetes type 1 diabetes may be an autoimmune disorder, but if well controlled one..."
6186,poor tissue status,"doctor: in brief: diabetes type 1 diabetes may be an autoimmune disorder, but if well controlled one..."
6187,infections,"doctor: in brief: diabetes type 1 diabetes may be an autoimmune disorder, but if well controlled one..."


In [24]:
problem_cases = df[
    df.apply(
        lambda row: (
            row["original_term"].strip().lower() == "diabetes"
            and bool(
                re.search(
                    r"type\s*(1|2|one|two|i|ii)\s+diabetes",
                    row["turn_text"][
                        max(0, row["char_start"] - 20):row["char_end"]
                    ],
                    re.IGNORECASE
                )
            )
        ),
        axis=1
    )
]

problem_cases[["original_term", "char_start", "char_end", "turn_text"]]

,original_term,char_start,char_end,turn_text
